# Back scratcher - record & repeat

You move the arm by hand, it **records** the movement, then it **repeats** it.

**Every time:** Kernel → Restart Kernel and Clear Outputs, then run cell 1 and cell 2 (Shift+Enter).

**■ Stop** = stop now (during Scratch it then goes to the sleep pose). Arm power plug = emergency stop.

⚠️ When recording, the motors go loose - **hold the arm** or it drops.

## 1. Settings

In [ ]:
import json, os, time

SIM = False            # True = test without the arm

MINUTES = 10           # how long it scratches, then goes to sleep
SPEED = 1.0            # 1.0 = same speed you recorded. 0.7 = slower/gentler, 1.3 = faster (max 1.5)
RECORD_SECONDS = 20    # max length of one recording (Stop button ends it earlier)
SAMPLE_S = 0.15        # how often it saves the position while recording

MOVE_FILE = os.path.expanduser("~/scratch_move.json")
SLEEP_FILE = os.path.expanduser("~/scratch_sleep.json")


## 2. Program + connect (just run it)

In [ ]:
SPEED = max(0.2, min(1.5, SPEED))

class SimArm:
    def __init__(self): self.a = [90, 90, 90, 90, 90, 150]
    def Arm_serial_servo_write6_array(self, a, t): self.a = list(a); print(f"  [sim] -> {a} ({t} ms)")
    def Arm_serial_servo_read(self, s): return self.a[s - 1]
    def Arm_serial_set_torque(self, on): print(f"  [sim] torque {'ON' if on else 'OFF'}")

def connect():
    if SIM:
        return SimArm()
    from Arm_Lib import Arm_Device
    a = Arm_Device()
    time.sleep(0.1)
    return a

def clamp(sid, v):
    return int(max(0, min(270 if sid == 5 else 180, v)))

_last = [90, 90, 90, 90, 90, 150]
def read_pose():
    """Read all servos; a failed read keeps the last good value."""
    global _last
    pose = list(_last)
    for sid in range(1, 7):
        for _ in range(4):
            try:
                v = arm.Arm_serial_servo_read(sid)
            except Exception:
                v = None
            if v is not None and 0 <= v <= (270 if sid == 5 else 180):
                pose[sid - 1] = int(v)
                break
            time.sleep(0.01)
    _last = pose
    return list(pose)

def go(pose, ms):
    pose = [clamp(i + 1, v) for i, v in enumerate(pose)]
    arm.Arm_serial_servo_write6_array(pose, int(ms))
    if not SIM:
        time.sleep(ms / 1000)

def load(path, what):
    if not os.path.exists(path):
        raise SystemExit(f"No {what} saved yet - run the 'Record {what}' cell first.")
    with open(path) as f:
        return json.load(f)

def sleep_pose():
    pose = load(SLEEP_FILE, "sleep pose")
    print("Going to sleep pose", pose)
    go(pose, 2500)

def record_sleep():
    print("HOLD THE ARM with your hand now - the motors are about to go loose!")
    time.sleep(0 if SIM else 3)
    arm.Arm_serial_set_torque(0)
    try:
        input("Put the arm in the SLEEP position, then click here and press Enter: ")
        pose = read_pose()
    finally:
        arm.Arm_serial_set_torque(1)
    go(pose, 500)  # hold it there
    with open(SLEEP_FILE, "w") as f:
        json.dump(pose, f)
    print("Sleep pose saved:", pose)

def record_move():
    print("HOLD THE ARM with your hand - motors go loose in 3 seconds.")
    print("Then do the scratching movement slowly, the way you want it.")
    print(f"Recording stops after {RECORD_SECONDS} s, or press the Stop button.")
    for s in (3, 2, 1):
        print(f"  {s}...")
        time.sleep(0 if SIM else 1)
    arm.Arm_serial_set_torque(0)
    frames = []
    print("  RECORDING - move it now!")
    t0 = time.time()
    try:
        while time.time() - t0 < (0.6 if SIM else RECORD_SECONDS):
            frames.append(read_pose())
            time.sleep(SAMPLE_S)
    except KeyboardInterrupt:
        pass
    finally:
        arm.Arm_serial_set_torque(1)
    if frames:
        go(frames[-1], 300)  # hold the last position
    # drop frames where nothing moved (keeps playback smooth)
    clean = [frames[0]] if frames else []
    for p in frames[1:]:
        if max(abs(a - b) for a, b in zip(p, clean[-1])) >= 2:
            clean.append(p)
    if len(clean) < 2:
        print("Hardly any movement recorded - try again and move the arm more.")
        return
    with open(MOVE_FILE, "w") as f:
        json.dump({"sample_s": SAMPLE_S, "frames": clean}, f)
    print(f"Saved movement: {len(clean)} steps, about {len(clean) * SAMPLE_S:.0f} s long.")

def play_once(data):
    step_ms = max(80, data["sample_s"] * 1000 / SPEED)
    go(data["frames"][0], 1500)          # slowly to the start
    for p in data["frames"][1:]:
        go(p, step_ms)

def test_once():
    data = load(MOVE_FILE, "movement")
    try:
        play_once(data)
    except KeyboardInterrupt:
        print("Stopped.")

def scratch():
    data = load(MOVE_FILE, "movement")
    load(SLEEP_FILE, "sleep pose")       # check it exists before starting
    print(f"Scratching for {MINUTES} min. Stop button = stop and go to sleep pose.")
    end = time.time() + MINUTES * 60
    n = 0
    try:
        while time.time() < end:
            play_once(data)
            n += 1
            if SIM and n >= 2:
                break
        print("Time's up.")
    except KeyboardInterrupt:
        print("Stopped.")
    finally:
        sleep_pose()

arm = connect()
print("Connected" if not SIM else "Simulation mode")


## Record sleep pose (once)
Hold the arm, put it where it should rest, click in the box, press Enter.

In [ ]:
record_sleep()

## Record movement (once, or again to change it)
After the countdown, do the scratching movement by hand, slowly. Do it on a pillow, the way it should touch your back. Gripper open/close counts too.

In [ ]:
record_move()

## Test: play the movement once
Watch it before lying down.

In [ ]:
test_once()

## Scratch!
Repeats the movement for `MINUTES`, then goes to sleep pose.

In [ ]:
scratch()

## Sleep now

In [ ]:
sleep_pose()